# SVAR-FGES Quickstart (Tetrad + Tigramite format)

This notebook is a minimal, reproducible test.
It installs required Python packages, generates synthetic data, runs SVAR-FGES, and returns a Tigramite-style graph.

In [ ]:
%pip install -q jpype1 numpy pandas

In [ ]:
import os
import sys
import numpy as np

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
SRC_PATH = os.path.join(PROJECT_ROOT, 'src')
UTILS_PATH = os.path.join(PROJECT_ROOT, 'utils')

for path in [SRC_PATH, UTILS_PATH]:
    if path not in sys.path:
        sys.path.insert(0, path)

from time_series_gen import generate_random_contemp_model, generate_nonlinear_contemp_timeseries
from svarfges import run_svarfges

In [ ]:
def lin_f(x):
    return x

seed = 42
rs = np.random.RandomState(seed)
N = 4
L = 8
lag_max = 2

links_coeffs = generate_random_contemp_model(
    N=N,
    L=L,
    coupling_coeffs=np.linspace(-0.4, 0.4, 9).tolist(),
    coupling_funcs=[lin_f],
    auto_coeffs=[0.3],
    tau_max=lag_max,
    contemp_fraction=0.3,
    random_state=rs,
)

T = 600
data, nonstationary = generate_nonlinear_contemp_timeseries(
    links_coeffs,
    T=T,
    random_state=rs,
)

if nonstationary:
    raise RuntimeError('Generated series is nonstationary. Try another seed.')

var_names = [f'X{i}' for i in range(N)]
data.shape

In [ ]:
result = run_svarfges(
    data=data,
    lag_max=lag_max,
    var_names=var_names,
    penalty_discount=2.0,
    faithfulness_assumed=True,
    symmetric_first_step=False,
    replicating=True,
    verbose=False,
    max_degree=-1,
    num_threads=1,
)

graph = result['graph']
val_matrix = result['val_matrix']

print('graph shape:', graph.shape)
print('val_matrix shape:', val_matrix.shape)
print('num non-empty marks:', int(np.sum(graph != '')))

In [ ]:
# Optional: inspect discovered links by lag
for tau in range(graph.shape[2]):
    idx = np.argwhere(graph[:, :, tau] != '')
    print(f'Lag {tau}: {len(idx)} links')
    for i, j in idx:
        print(f'  {var_names[i]} -(tau={tau})-> {var_names[j]} : {graph[i, j, tau]}')